DataFrame and Datasets are table like collection with well-defined rows and column, when we perform an action on dataframe, we instruct spark
to perform the acutal transformations and return the results.

### Schemas :
  A schema defines the column names and types of a DataFrame.
  you can define a schema manually or read a schema from a data source ( often called schema on read)

### Spark Types:
  Even you write you code in scala or python, internally spark usese optimization and does perform operion on spark types.

In [ ]:
from pyspark.sql.session import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-type")
    .master("local[*]")
    .config("spark.port.maxRetries", "0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/02 16:23:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
from pyspark.sql.functions import col

In [10]:
df = spark.range(500).toDF("number")

df.select(col("number") + 10).show(10,False)


+-------------+
|(number + 10)|
+-------------+
|10           |
|11           |
|12           |
|13           |
|14           |
|15           |
|16           |
|17           |
|18           |
|19           |
+-------------+
only showing top 10 rows


This addition operation happens because Spark will convert an expression written in an input
language to Spark’s internal Catalyst representation of that same type information

## **DataFrames Versus Datasets**

DataFrames are
simply Datasets of Type Row. The “Row” type is Spark’s internal representation of its optimized
in-memory format for computation. This format makes for highly specialized and efficient
computation because rather than using JVM types, which can cause high garbage-collection and
object instantiation costs, Spark can operate on its own internal format without incurring any of
those costs.

#### **Columns**
Columns represent a simple type like an integer or string, a complex type like an array or map, or a null value

- Spark tracks all of this type information for you and offers a variety of ways, with
which you can transform columns

#### **Rows**
A row is nothing more than a record of data. Each record in a DataFrame must be of type Row.



In [12]:
spark.range(2).toDF("number").collect()

[Row(number=0), Row(number=1)]

#### **Spark Types**
Spark has a large number of internal type representations.

In [ ]:
from pyspark.sql.types import ByteType, LongType, IntegerType, ShortType
from pyspark.sql.types import StringType, BooleanType, DateType, ArrayType
from pyspark.sql.types import StructField, StructType, MapType

#### **overview of structed api execution**

1. Write DataFrame/Dataset/SQL Code.
2. If valid code, Spark converts this to a Logical Plan.
3. Spark transforms this Logical Plan to a Physical Plan, checking for optimizations along
the way.
4. Spark then executes this Physical Plan (RDD manipulations) on the cluster.

<img src="../data/pictures/catelyst_optimizer_01.png" width="600">


<img src="../data/pictures/catelyst_optimizer_02.png" width="600">

spark takes the user code and create a these plan step by step

1. **unresolved logical plan** : This plan is unresolved
because although your code might be valid, the tables or columns that it refers to might or might not exist

2. **resolved logical plan**:  spark **analyzer**  create the resolved logical plan by using **catalog** a repository of all table and DataFrame information, to resolve columns and tables in the analyzer. The **Analyzer** might reject the unresolved logical plan if the required table or column name does not exist in the catalog.

3. **Catalyst Optimizer** : if the analyzer can resolve the plan then result is passed throught the Catalyst optimizer ( a collection of ruls that attempt to optimize the logical plan by pushing down predicates or selection)

4. **Physical Planning** : After successfully creating an optimized logical plan, Spark then begins the physical planning
process. The physical plan, often called a Spark plan.

Physical planning results in a series of RDDs and transformations.


5. **Execution**:  Spark runs all of this code over RDDs, the lower-level
programming interface of Spark (which we cover in Part III). Spark performs further
optimizations at runtime, generating native Java bytecode that can remove entire tasks or stages
during execution. Finally the result is returned to the user.

In [22]:
spark.stop()